# 12 · Performance: plans, partitions, cost (BigQuery)

Each engine is fast in a different way, so this notebook has **different questions per engine**.

- **BigQuery** charges by **bytes read**. Columnar storage (each column stored apart): read fewer columns = pay less.
  **Partitioning** (split a table by date) and **clustering** (sort inside partitions) let it skip data.
- **Oracle** uses an **execution plan** made by the cost-based optimizer: full table scan, index, join method.
  `EXPLAIN PLAN` shows it.
- **Spark** builds a plan too (`explain()`). Look for **pushed filters**, **partition filters**,
  **broadcast joins** and **exchanges** (shuffles: data moving between machines, the slow part).

The tables here are tiny, so speed differences are small. Learn to **read** the plans: that is what interviews ask.

**How to run:** Start each SQL cell with `%%bigquery`. Write tables as `practice.table_name`.
Each cell prints how much data it read. `%%bigquery dry` checks the size for free.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`bigquery/practice/12_performance.ipynb`](../../bigquery/practice/12_performance.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "bigquery"))
%load_ext bq_helper

## Q1. What will this query cost? (dry run)  ·  🟢 easy

Use `%%bigquery dry` (free) to see how many bytes these read: (1) `SELECT *` from orders, (2) only `order_id, amount`, (3) `SELECT *` with `LIMIT 5`.

**How it works:** A dry run plans the query without running it and reports the bytes it would read. Compare the numbers.

In [ ]:
%%bigquery dry
SELECT * FROM practice.orders

Would read 5.3 KB

In [ ]:
%%bigquery dry
SELECT order_id, amount FROM practice.orders

Would read 2.4 KB

In [ ]:
%%bigquery dry
SELECT * FROM practice.orders LIMIT 5

Would read 5.3 KB

**Why / traps:** Fewer columns = fewer bytes. **LIMIT does not reduce the bytes billed**: BigQuery still reads the whole columns. (Every query is billed at least 10 MB.)

## Q2. Create a partitioned and clustered table  ·  🟡 medium

Create `practice.prac_orders_part` from the clean orders, **partitioned by month** of order_date and **clustered by** customer_id. Then list its first partitions from `INFORMATION_SCHEMA.PARTITIONS`.

**How it works:** CREATE TABLE ... PARTITION BY (month) CLUSTER BY (customer_id) AS SELECT ...; INFORMATION_SCHEMA.PARTITIONS lists the partitions and their rows.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE practice.prac_orders_part
PARTITION BY DATE_TRUNC(order_date, MONTH)
CLUSTER BY customer_id
AS SELECT DISTINCT * FROM practice.orders

OK (create table as select)

In [ ]:
%%bigquery
SELECT partition_id, total_rows
FROM practice.INFORMATION_SCHEMA.PARTITIONS
WHERE table_name = 'prac_orders_part'
ORDER BY partition_id
LIMIT 6

,partition_id,total_rows
0,202303,1
1,202306,2
2,202308,1
3,202310,1
4,202311,3
5,202312,2


**Why / traps:** Partition by the column you **filter on most** (usually a date). Cluster by columns used in filters/joins with many values (customer_id). Monthly partitions keep the partition count low for small data; daily is common for big tables (max 10,000 partitions per table).

## Q3. Partition pruning  ·  🟡 medium

Dry-run 3 filters on `prac_orders_part` and compare the bytes: (1) no filter, (2) `order_date` between 2025-10-01 and 2025-10-31, (3) `EXTRACT(YEAR FROM order_date) = 2025`.

**How it works:** Dry-run the same query with different filters. Fewer bytes = BigQuery skipped partitions.

In [ ]:
%%bigquery dry
SELECT order_id, amount FROM practice.prac_orders_part

Would read 2.3 KB

In [ ]:
%%bigquery dry
SELECT order_id, amount FROM practice.prac_orders_part
WHERE order_date BETWEEN DATE '2025-10-01' AND DATE '2025-10-31'

Would read 128 B

In [ ]:
%%bigquery dry
SELECT order_id, amount FROM practice.prac_orders_part
WHERE EXTRACT(YEAR FROM order_date) = 2025

Would read 1.7 KB

**Why / traps:** **Partition pruning** = skipping partitions that cannot match. A plain range on the partition column prunes. Wrapping the column in a function may stop pruning. For safety on big tables set `require_partition_filter = TRUE` so a query without a date filter fails instead of costing a lot.

## Q4. Exact vs approximate distinct count  ·  🟢 easy

Count different customers in `orders` two ways: `COUNT(DISTINCT customer_id)` and `APPROX_COUNT_DISTINCT(customer_id)`.

**How it works:** Run both counts side by side; on small data they match, on big data the approximate one is much cheaper.

In [ ]:
%%bigquery
SELECT COUNT(DISTINCT customer_id)        AS exact_customers,
       APPROX_COUNT_DISTINCT(customer_id) AS approx_customers
FROM practice.orders

,exact_customers,approx_customers
0,32,32


**Why / traps:** On billions of rows the approximate version (HyperLogLog++) is much faster and uses little memory, with ~1% error. Same idea: Oracle `APPROX_COUNT_DISTINCT`, Spark `approx_count_distinct`.

## Q5. What did my queries cost?  ·  🟡 medium

List your last 5 query jobs with `bytes_billed` from `INFORMATION_SCHEMA.JOBS_BY_USER`.

Use **your dataset's region** in the name, e.g. `region-us`, `region-eu`, `region-europe-west2`.

**How it works:** `INFORMATION_SCHEMA.JOBS_BY_USER` keeps your query history with the bytes billed for each job.

In [ ]:
%%bigquery
SELECT creation_time, statement_type, total_bytes_billed AS bytes_billed,
       SUBSTR(query, 1, 60) AS query_start
FROM `region-europe-west2`.INFORMATION_SCHEMA.JOBS_BY_USER
WHERE job_type = 'QUERY'
ORDER BY creation_time DESC
LIMIT 5

,creation_time,statement_type,bytes_billed,query_start
0,2026-11-14 22:24:46.576000+00:00,SELECT,10485760,SELECT COUNT(DISTINCT customer_id) AS exact_customers
1,2026-11-14 22:24:43.122000+00:00,SELECT,10485760,"SELECT partition_id, total_rows\nFROM practice.INFORMATION_SC"
2,2026-11-14 22:24:40.554000+00:00,CREATE_TABLE_AS_SELECT,10485760,CREATE OR REPLACE TABLE practice.prac_orders_part\nPARTITION
3,2026-11-14 22:24:37.160000+00:00,SELECT,10485760,"SELECT DISTINCT order_id, customer_id, amount\nFROM practice."
4,2026-11-14 22:24:35.469000+00:00,SELECT,10485760,"SELECT MAX(order_date) AS latest_order_date,\n DATE_DIF"


**Why / traps:** Watch this when you learn: it shows which queries were expensive. Cached results cost 0.